# Week 1 — NVIDIA NIM PDF 질문 답변 평가

AIHub PDF 전처리부터 NVIDIA NIM 실제 40건, Pydantic과 DeepEval 평가, recorded 회귀까지 순서대로 실행합니다. 실제 API가 이 실습의 중심이며 저장 응답은 마지막 회귀 단계에서 사용합니다.

## 1. 환경과 key

프로젝트 루트에서 먼저 실행합니다.

```bash
uv sync --locked --dev
cp .env.example .env
uv run jupyter lab
```

`.env`의 `NVIDIA_NIM_API_KEY`에 발급받은 key를 넣습니다. Notebook에는 key를 직접 쓰지 않습니다.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import JSON, Image, display

from verifiable_ai_workflow.config import load_project_env, load_settings, project_path
from verifiable_ai_workflow.data.dataset import build_cases, write_cases
from verifiable_ai_workflow.preprocessing import load_document, prepare_directory
from verifiable_ai_workflow.schemas import StructuredAnswer

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트에서 JupyterLab을 실행해 주세요.")
os.chdir(PROJECT_ROOT)
load_project_env(PROJECT_ROOT)
print("project:", PROJECT_ROOT)
print("NVIDIA key configured:", bool(os.getenv("NVIDIA_NIM_API_KEY")))

## 2. PDF와 질문 준비

`local-data/aihub/source/`에 AIHub sample을 넣은 뒤 전체 페이지를 PNG, API용 JPEG와 text로 준비합니다.

In [ ]:
settings = load_settings(PROJECT_ROOT / "configs/week-01.yaml")
manifests = prepare_directory(
    project_path(PROJECT_ROOT, settings.paths.raw_documents),
    project_path(PROJECT_ROOT, settings.paths.prepared_documents),
    render_dpi=settings.documents.render_dpi,
    model_image_max_bytes=settings.documents.model_image_max_bytes,
    model_image_max_width=settings.documents.model_image_max_width,
)
cases = build_cases(project_path(PROJECT_ROOT, settings.paths.case_authoring))
write_cases(cases, project_path(PROJECT_ROOT, settings.paths.cases))
print("documents:", len(manifests), "cases:", len(cases))

In [ ]:
document, manifest_path = load_document(
    project_path(PROJECT_ROOT, settings.paths.prepared_documents),
    cases[0].document_id,
)
print("pages:", document.total_pages)
display(Image(filename=str(manifest_path.parent / document.pages[0].image_path), width=550))

## 3. PDF·질문·prompt EDA

문서·질문 분포, 기대 페이지 범위, API 이미지 크기, PDF text와 prompt field를 확인합니다.

In [ ]:
subprocess.run([sys.executable, "scripts/inspect_inputs.py"], check=True)
eda = json.loads((PROJECT_ROOT / "reports/week-01/eda.json").read_text(encoding="utf-8"))
display(
    JSON(
        {
            "document_count": eda["document_count"],
            "case_count": eda["case_count"],
            "answer_type_counts": eda["answer_type_counts"],
            "label_text_check_counts": eda["label_text_check_counts"],
            "anomalies": eda["anomalies"],
        }
    )
)

## 4. 구조화 응답 계약

`StructuredAnswer`는 answer, evidence, confidence, abstained, abstention_reason, tool_requests를 검사합니다. 일반 답변은 근거가 필요하고 답변 보류에는 근거를 넣지 않습니다.

In [ ]:
display(JSON(StructuredAnswer.model_json_schema()))
print((PROJECT_ROOT / "prompts/pdf-question-answer.md").read_text(encoding="utf-8"))

## 5. NVIDIA model preflight

웹 카탈로그 표시와 실제 endpoint 상태가 다를 수 있으므로 실제 `/v1/models`에서 설정 모델을 확인합니다.

In [ ]:
subprocess.run([sys.executable, "scripts/preflight_nvidia.py"], check=True)

## 6. 실제 첫 1건

이 호출은 40건 중 첫 번째입니다. raw response를 즉시 저장하고 Pydantic과 정량 metric을 실행합니다.

In [ ]:
live_observations = PROJECT_ROOT / "reports/week-01-nvidia/observations.jsonl"
command = [sys.executable, "scripts/run_nvidia_nim.py", "--live", "--limit", "1"]
if live_observations.exists():
    print("기존 실행이 있습니다. 첫 호출을 건너뜁니다:", live_observations)
else:
    subprocess.run(command, check=True)

In [ ]:
first_observation = json.loads(live_observations.read_text(encoding="utf-8").splitlines()[0])
first_result = json.loads(
    (PROJECT_ROOT / "reports/week-01-nvidia/results.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()[0]
)
display(
    JSON(
        {
            "sample_id": first_observation["sample_id"],
            "raw_output": first_observation["raw_output"],
            "model_error": first_observation["model_error"],
            "model_call": first_observation["model_call"],
            "status": first_result["status"],
            "scores": first_result["scores"],
        }
    )
)

## 7. 나머지 39건 순차 실행

20 RPM 이하, 429 backoff와 중간 저장을 적용합니다. 완료된 sample은 건너뜁니다.

In [ ]:
subprocess.run(
    [sys.executable, "scripts/run_nvidia_nim.py", "--live", "--resume"],
    check=True,
)
summary = json.loads(
    (PROJECT_ROOT / "reports/week-01-nvidia/summary.json").read_text(encoding="utf-8")
)
display(JSON(summary))

## 8. 실제 응답 고정과 회귀평가

40건 전체에 provider 오류가 없고 raw response 고정을 별도로 승인받은 경우에만 recorded fixture로 고정합니다.

In [ ]:
subprocess.run([sys.executable, "scripts/freeze_recorded_responses.py"], check=True)
subprocess.run([sys.executable, "scripts/run_workflow.py"], check=True)
subprocess.run([sys.executable, "scripts/evaluate_workflow.py"], check=True)

## 9. 실패 주입

깨진 JSON, confidence 범위 위반, 오답과 잘못된 페이지가 예상 metric을 실패시키는지 확인합니다.

In [ ]:
subprocess.run([sys.executable, "scripts/evaluate_failures.py"], check=True)
display(
    JSON(
        json.loads(
            (PROJECT_ROOT / "reports/week-01-failures/results.json").read_text(encoding="utf-8")
        )
    )
)

## 완료

`reports/week-01-nvidia/summary.json`에서 40건의 passed, failed, inconclusive와 exact, ANLS, token F1, 숫자, 근거 페이지·인용 점수를 확인합니다. DeepEval TUI는 `uv run deepeval inspect reports/week-01-nvidia/deepeval`로 엽니다.